# Attention Head Diversity

How different heads within a layer behave: pattern diversity,
output diversity, entropy variation, and focus diversity.

## Why This Matters

Attention head diversity reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_diversity import (
    pattern_diversity, output_diversity,
    entropy_diversity, attention_focus_diversity,
    head_diversity_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Pattern Diversity

Are attention patterns diverse or redundant across heads?

In [ ]:
for layer in range(4):
    result = pattern_diversity(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_diverse'] else ''
    print(f"  Layer {layer}: sim={result['mean_pattern_similarity']:.4f}{flag}")
    for p in result['head_pairs'][:3]:
        print(f"    H{p['head_a']}-H{p['head_b']}: {p['similarity']:.4f}")

## Output Diversity

Do heads produce diverse or redundant outputs?

In [ ]:
for layer in range(4):
    result = output_diversity(model, tokens, layer=layer, position=-1)
    flag = ' [DIVERSE]' if result['is_diverse'] else ''
    print(f"  Layer {layer}: sim={result['mean_output_similarity']:.4f}{flag}")

## Entropy Diversity

Do heads have different sharpness levels?

In [ ]:
for layer in range(4):
    result = entropy_diversity(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_entropy_diverse'] else ''
    print(f"  Layer {layer}: range={result['entropy_range']:.4f}{flag}")
    for h in result['per_head']:
        bar = '█' * int(h['mean_entropy'] * 5)
        print(f"    H{h['head']}: H={h['mean_entropy']:.4f} {bar}")

## Focus Diversity

Do heads attend to different positions?

In [ ]:
for layer in range(4):
    result = attention_focus_diversity(model, tokens, layer=layer)
    print(f"  Layer {layer}: diversity={result['focus_diversity']:.4f}, "
          f"mean_unique={result['mean_unique_focuses']:.2f}")

## Diversity Summary

In [ ]:
result = head_diversity_summary(model, tokens)
for p in result['per_layer']:
    flags = []
    if p['is_pattern_diverse']: flags.append('PAT')
    if p['is_entropy_diverse']: flags.append('ENT')
    flag_str = f" [{', '.join(flags)}]" if flags else ''
    print(f"  Layer {p['layer']}: pat_sim={p['pattern_similarity']:.4f}, "
          f"ent_range={p['entropy_range']:.4f}{flag_str}")